# Evaluation (Step 3, Day 3) — Field Accuracy vs Gold

Measures how well each OCR→LLM pipeline matches human gold labels.

**Inputs:**
- `gold/*.json` — hand-labeled ground truth (see `gold/README.md`)
- `artifacts/extracted/marriage_records_docling.jsonl`
- `artifacts/extracted/marriage_records_trocr.jsonl`

**Outputs:**
- Per-field accuracy for each OCR source
- Head-to-head table: which engine feeds better structured extraction
- Error breakdown (missed vs wrong vs hallucinated)

Run this after you've labeled a handful of pages in `gold/`."

In [2]:
# 1. Config & loaders
import json
from pathlib import Path

import pandas as pd

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

GOLD_DIR = PROJECT_ROOT / "gold"
EXTRACTED_DIR = PROJECT_ROOT / "artifacts/extracted"

# Flat, dotted field paths we score. Provenance keys are excluded.
FIELDS = [
    "license_number", "application_number",
    "marriage_day", "marriage_month", "marriage_year", "marriage_location",
    "groom.full_name", "groom.residence", "groom.age", "groom.color",
    "groom.nativity", "groom.occupation", "groom.marital_status",
    "bride.full_name", "bride.residence", "bride.age", "bride.color",
    "bride.nativity", "bride.occupation", "bride.marital_status",
    "relationship_between_parties", "officiant.signature",
]


def get_field(rec: dict, dotted: str):
    cur = rec
    for part in dotted.split("."):
        cur = cur.get(part) if isinstance(cur, dict) else None
    return cur


def load_jsonl(path: Path) -> dict[tuple[str, int], dict]:
    out: dict[tuple[str, int], dict] = {}
    if not path.exists():
        return out
    for line in path.open():
        line = line.strip()
        if line:
            r = json.loads(line)
            out[(r["source_pdf"], r["source_page"])] = r
    return out


def load_gold() -> dict[tuple[str, int], dict]:
    out: dict[tuple[str, int], dict] = {}
    for jp in sorted(GOLD_DIR.glob("*.json")):
        if jp.name.startswith("_"):  # skip _template.json
            continue
        r = json.loads(jp.read_text())
        out[(r["source_pdf"], r["source_page"])] = r
    return out


gold = load_gold()
docling = load_jsonl(EXTRACTED_DIR / "marriage_records_docling.jsonl")
trocr = load_jsonl(EXTRACTED_DIR / "marriage_records_trocr.jsonl")

print(f"gold: {len(gold)} labeled pages")
print(f"docling extractions: {len(docling)}")
print(f"trocr extractions:   {len(trocr)}")
if not gold:
    print("\n>>> No gold labels yet. Label pages in gold/ (see gold/README.md).")

gold: 14 labeled pages
docling extractions: 225
trocr extractions:   225


In [3]:
# 2. Scoring helpers
import re
from difflib import SequenceMatcher

# Free-text / handwritten fields where minor spelling drift is acceptable.
# These use fuzzy matching instead of exact match.
FUZZY_FIELDS = {
    "officiant.signature",
    "groom.full_name", "bride.full_name",
    "groom.residence", "bride.residence",
    "groom.nativity", "bride.nativity",
    "groom.occupation", "bride.occupation",
}
FUZZY_THRESHOLD = 0.85  # SequenceMatcher ratio at/above which we call it correct


def norm(v) -> str | None:
    """Normalize for comparison: uppercase, drop punctuation, collapse whitespace.

    So 'BALTIMORE MD.' == 'BALTIMORE MD', 'S.C.' == 'SC',
    and 'G. M. Harris' == 'G M Harris'. None stays None.
    """
    if v is None:
        return None
    s = str(v).upper()
    s = re.sub(r"[.,;:'\"()]", " ", s)   # punctuation -> space
    s = re.sub(r"\s+", " ", s).strip()    # collapse whitespace
    return s or None


def similar(g: str, p: str) -> float:
    """Fuzzy similarity in [0, 1] on normalized, word-order-insensitive strings."""
    gt = " ".join(sorted(g.split()))
    pt = " ".join(sorted(p.split()))
    return SequenceMatcher(None, gt, pt).ratio()


def classify(gold_v, pred_v, field: str | None = None) -> str:
    """Compare one field: correct / missed / wrong / hallucinated / true_null.

    For FUZZY_FIELDS (names, signatures, residence, ...) a normalized
    similarity >= FUZZY_THRESHOLD counts as correct, so punctuation and
    minor spelling drift aren't penalized as wrong.
    """
    g, p = norm(gold_v), norm(pred_v)
    if g is not None and p is not None:
        if g == p:
            return "correct"
        if field in FUZZY_FIELDS and similar(g, p) >= FUZZY_THRESHOLD:
            return "correct"
        return "wrong"
    if g is not None and p is None:
        return "missed"           # value existed, model left it null
    if g is None and p is not None:
        return "hallucinated"     # model invented a value
    return "true_null"            # both null — correctly empty


def score_source(name: str, preds: dict) -> pd.DataFrame:
    """Per-field outcome counts for pages present in BOTH gold and preds."""
    keys = sorted(set(gold) & set(preds))
    rows = []
    for field in FIELDS:
        counts = {"correct": 0, "wrong": 0, "missed": 0,
                  "hallucinated": 0, "true_null": 0}
        for k in keys:
            counts[classify(get_field(gold[k], field),
                            get_field(preds[k], field), field)] += 1
        # Accuracy over fields that HAVE a gold value (exclude true_null).
        gold_present = counts["correct"] + counts["wrong"] + counts["missed"]
        acc = counts["correct"] / gold_present if gold_present else float("nan")
        rows.append({"field": field, "source": name,
                     "accuracy": acc, **counts, "n_gold": gold_present})
    return pd.DataFrame(rows)


print("Scoring helpers ready.")

Scoring helpers ready.


In [4]:
# 3. Headline: overall accuracy per OCR source
assert gold, "Label some pages in gold/ first (see gold/README.md)."

doc_scores = score_source("docling", docling)
tro_scores = score_source("trocr", trocr)


def overall(df: pd.DataFrame) -> dict:
    tot = df[["correct", "wrong", "missed"]].sum()
    gold_present = int(tot.sum())
    return {
        "field_accuracy": tot["correct"] / gold_present if gold_present else float("nan"),
        "correct": int(tot["correct"]),
        "wrong": int(tot["wrong"]),
        "missed": int(tot["missed"]),
        "hallucinated": int(df["hallucinated"].sum()),
    }


summary = pd.DataFrame({"docling": overall(doc_scores), "trocr": overall(tro_scores)}).T
n_pages = len(set(gold) & set(docling))
print(f"Evaluated on {n_pages} gold-labeled pages, {len(FIELDS)} fields each.\n")
summary.style.format({"field_accuracy": "{:.1%}"})

Evaluated on 14 gold-labeled pages, 22 fields each.



,field_accuracy,correct,wrong,missed,hallucinated
docling,57.1%,168.000000,73.000000,53.000000,0.000000
trocr,54.1%,159.000000,82.000000,53.000000,0.000000


In [5]:
# 4. Per-field accuracy, docling vs trocr side by side
merged = doc_scores[["field", "accuracy", "n_gold"]].merge(
    tro_scores[["field", "accuracy"]], on="field", suffixes=("_docling", "_trocr")
)
merged["delta"] = merged["accuracy_docling"] - merged["accuracy_trocr"]
merged = merged.sort_values("accuracy_docling", ascending=False)

(merged.style
    .format({"accuracy_docling": "{:.0%}", "accuracy_trocr": "{:.0%}", "delta": "{:+.0%}"})
    .background_gradient(subset=["delta"], cmap="RdYlGn", vmin=-1, vmax=1))

,field,accuracy_docling,n_gold,accuracy_trocr,delta
8,groom.age,100%,14,100%,+0%
6,groom.full_name,93%,14,93%,+0%
11,groom.occupation,86%,14,79%,+7%
19,bride.marital_status,86%,14,86%,+0%
18,bride.occupation,86%,14,79%,+7%
13,bride.full_name,86%,14,86%,+0%
7,groom.residence,79%,14,71%,+7%
14,bride.residence,79%,14,79%,+0%
12,groom.marital_status,71%,14,71%,+0%
15,bride.age,64%,14,64%,+0%


In [6]:
# 5. Inspect actual errors (great for the interview walkthrough)
def error_rows(name: str, preds: dict, kinds=("wrong", "missed", "hallucinated")):
    keys = sorted(set(gold) & set(preds))
    rows = []
    for k in keys:
        for field in FIELDS:
            outcome = classify(get_field(gold[k], field), get_field(preds[k], field), field)
            if outcome in kinds:
                rows.append({
                    "page": f"{k[0]} p{k[1]}", "field": field, "outcome": outcome,
                    "gold": get_field(gold[k], field),
                    name: get_field(preds[k], field),
                })
    return pd.DataFrame(rows)


print("Docling errors vs gold:")
error_rows("docling", docling)

Docling errors vs gold:


,page,field,outcome,gold,docling
0,1_content.pdf p1,marriage_day,wrong,20,30
1,1_content.pdf p1,groom.marital_status,missed,DIVORCED,None
2,1_content.pdf p1,bride.age,missed,23,None
3,1_content.pdf p1,bride.color,missed,COL,None
4,1_content.pdf p1,bride.nativity,missed,NY,None
...,...,...,...,...,...
121,2_content.pdf p90,groom.nativity,wrong,MD,TNativity
122,2_content.pdf p90,bride.color,missed,WHT,None
123,2_content.pdf p90,bride.nativity,missed,MD,None
124,2_content.pdf p90,relationship_between_parties,wrong,NONE,(State here whether the contracting parties ar...


In [7]:
# 6. Export evaluation results to files
EVAL_DIR = PROJECT_ROOT / "artifacts/eval"
EVAL_DIR.mkdir(parents=True, exist_ok=True)

# Headline summary (docling vs trocr): field_accuracy, correct/wrong/missed/hallucinated
summary.to_csv(EVAL_DIR / "summary.csv")

# Per-field accuracy, docling vs trocr side by side (with delta)
merged.to_csv(EVAL_DIR / "per_field_accuracy.csv", index=False)

# Full per-field outcome breakdown for each source
full_scores = pd.concat([doc_scores, tro_scores], ignore_index=True)
full_scores.to_csv(EVAL_DIR / "per_field_breakdown.csv", index=False)

# Every individual error (wrong / missed / hallucinated) for both sources
errors = pd.concat(
    [
        error_rows("docling", docling).assign(source="docling").rename(columns={"docling": "prediction"}),
        error_rows("trocr", trocr).assign(source="trocr").rename(columns={"trocr": "prediction"}),
    ],
    ignore_index=True,
)
errors.to_csv(EVAL_DIR / "errors.csv", index=False)

# Machine-readable headline for reports / slides
summary.to_json(EVAL_DIR / "summary.json", orient="index", indent=2)

print(f"Wrote evaluation results to {EVAL_DIR}:")
for f in sorted(EVAL_DIR.glob("*")):
    print(f"  {f.name}")

Wrote evaluation results to /Users/akhileshmadyastha/Desktop/projects/document-understanding/artifacts/eval:
  errors.csv
  per_field_accuracy.csv
  per_field_breakdown.csv
  summary.csv
  summary.json
